In [ ]:
import pandas as pd
import numpy as np

class ZoneModel:
    def __init__(self, zone_name, **kwargs):
        self.zone_name = zone_name
        
        # --- Unpack all inputs from the orchestrator ---
        # Dataframe storage
        self.hourly_df = kwargs.get('hourly_df') # Storing the DF here
        
        # Schedules
        self.occ_profile = kwargs.get('occ_profile')
        self.equip_profile = kwargs.get('equip_profile')
        self.vent_profile = kwargs.get('vent_profile')
        self.system_profile = kwargs.get('system_profile')

        # Geometry
        self.floor_area = kwargs.get('floor_area')
        self.room_volume = kwargs.get('room_volume')
        self.area_roof = kwargs.get('area_roof')
        self.area_ground = kwargs.get('area_ground')
        self.total_facade_areas = kwargs.get('total_facade_areas')
        self.window_percentages = kwargs.get('window_percentages')
        self.glazing_percentages = kwargs.get('glazing_percentages')

        # Thermal Properties
        self.rc_facade = kwargs.get('rc_facade')
        self.rc_roof = kwargs.get('rc_roof')
        self.rc_ground_floor = kwargs.get('rc_ground_floor')
        self.u_value_windows = kwargs.get('u_value_windows')

        # Solar & Ventilation Constants
        self.alfai = kwargs.get('alfai')
        self.alfao = kwargs.get('alfao')
        
        self.solar_absorption_coefficient = kwargs.get('solar_absorption_coefficient')
        self.solar_heat_coefficient_shading = kwargs.get('solar_heat_coefficient_shading')
        self.solar_heat_coefficient_glazing = kwargs.get('solar_heat_coefficient_glazing')

        self.air_density = kwargs.get('air_density')
        self.air_heat_capacity = kwargs.get('air_heat_capacity')
        self.vent_flow_per_person = kwargs.get('vent_flow_per_person')
        self.infiltration_ach = kwargs.get('infiltration_ach')
        self.natural_vent_rate = kwargs.get('natural_vent_rate')

        # Internal Gains & System Constants
        self.max_people_per_m2 = kwargs.get('max_people_per_m2')
        self.heat_per_person = kwargs.get('heat_per_person')
        self.appliances_w_m2 = kwargs.get('appliances_w_m2')
        self.lighting_w_m2 = kwargs.get('lighting_w_m2')

        # --- HVAC & Ventilation System ---
        self.system_pressure_drop = kwargs.get('system_pressure_drop')
        self.efficiency_fan_and_motor = kwargs.get('efficiency_fan_and_motor')
        self.eta = kwargs.get('eta', 0.0) 

        # Simulation Constants
        self.heating_setpoint = kwargs.get('heating_setpoint')
        self.cooling_setpoint = kwargs.get('cooling_setpoint')

        self.vent_cooling_setpoint = kwargs.get('vent_cooling_setpoint')

        self.h_setpoint_profile = kwargs.get('heating_setpoint_profile')
        self.c_setpoint_profile = kwargs.get('cooling_setpoint_profile')
        self.vent_comfort_limit = kwargs.get('vent_comfort_limit', 23.5)

        self.thermal_mass_factor = kwargs.get('thermal_mass_factor', 165000) # Default to heavy if not provided
        self.thermal_capacity = self.thermal_mass_factor * self.floor_area
        self.dt = 3600
        self.t_ground = kwargs.get('t_ground')

        self.heating_power_max = kwargs.get('heating_power_max')
        self.cooling_power_max = kwargs.get('cooling_power_max')

        # Simulation starting temperature
        self.current_temp = kwargs.get('initial_temp', 21.0) # Need a starting temp

        # --- NEW: Map floor heating inputs ---
        self.u_floor_heating = kwargs.get('u_floor_heating', 11.0)
        self.floor_heating_area = kwargs.get('floor_heating_area', 0.0)
        self.t_floor_water_heating = kwargs.get('t_floor_water_heating', 35.0)
        self.t_floor_water_cooling = kwargs.get('t_floor_water_cooling', 18.0)
        self.floor_heating_activated = kwargs.get('floor_heating_activated', False)
        
        # --- NEW: Comfort limits for floor activation ---  
        self.t_floor_limit_heating = kwargs.get('t_floor_limit_heating', 22.0)
        self.t_floor_limit_cooling = kwargs.get('t_floor_limit_cooling', 20.0)

        # Pre-calculate surface areas for the loop
        self.solar_orientations = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
        self._initialize_surfaces()
        
    def _initialize_surfaces(self):
        self.area_windows = 0
        self.area_walls = 0
        self.glass_areas = {}
        self.opaque_wall_areas = {}

        for orient in self.solar_orientations:
            facade = self.total_facade_areas.get(orient, 0.0)
            win_perc = self.window_percentages.get(orient, 0.0)
            glaz_perc = self.glazing_percentages.get(orient, 1.0)
    
            window_area = facade * win_perc
            self.glass_areas[orient] = window_area * glaz_perc
            self.opaque_wall_areas[orient] = facade - window_area
            
            self.area_windows += window_area
            self.area_walls += self.opaque_wall_areas[orient]
            
        # Extra required calculations
        self.max_people = self.floor_area * self.max_people_per_m2
        self.flow_rate_vent = self.max_people * self.vent_flow_per_person # The same as total_flow_rate_vent
        self.total_flow_rate_vent = self.vent_flow_per_person * self.max_people_per_m2 * self.floor_area # The same as flow_rate_vent
        self.fans_power = (self.total_flow_rate_vent * self.system_pressure_drop) / (3600 * self.efficiency_fan_and_motor)
        
        # Tracking lists for results (outside loop to avoid reinitialization, since this simulation is per timestep)
        self.annual_heating_results = []
        self.annual_cooling_results = []
        self.annual_temp_results = []

        # Extended tracking for detailed hourly Excel export
        self.detailed_results = {
            "Hour": [], "T_ext": [],
            "Trans window": [], "Trans wall": [], "Trans roof": [], "Transmission tot (W/K)": [],
            "Infiltration (W/K)": [], "Ventilation heat mode (W/K)": [], "Ventilation cool mode (W/K)": [],
            "Heating mode tot coupling (W/K)": [], "Cooling mode tot coupling (W/K)": [], "Ground coupling (W/K)": [],
            "Solar glazing (Wh)": [], "Solar opaque (Wh)": [],
            "People (Wh)": [], "Lighting (Wh)": [], "Appliances (Wh)": [], "Fans (Wh)": [], "Internal heat gains (Wh)": [],
            "T_i Max heating": [], "T_i free floating with H_heat": [], "T_i Max cooling": [], "T_i free floating with H_cool": [],
            "T_i indoor (oC)": [], "Q_heat (kWh)": [], "Q_cool (kWh)": [],
            "Sch_Occupancy": [], "Sch_Equipment": [], "Sch_SystemActive": [], "Sch_Ventilation": []
        }

        # Add orientation-specific solar columns
        for orient in self.solar_orientations:
            self.detailed_results[f"G_{orient}_(Wh)"] = []
        for orient in self.solar_orientations:
            self.detailed_results[f"O_{orient}_(Wh)"] = []

    def calculate_nta8800_u_value(self, rc_value):
        if rc_value <= 0: return 0
        delta_u = 0.15 # Either use delta_u = max(0, 0.1 - 0.25 * (u_undisturbed - 0.4)) for accuracy, or the 0.15 fixed penalty as is done in the excel.
        return (1 / (rc_value + 0.17)) + delta_u

    def calculate_hour_step(self, t, external_q_flow=0, t_source=None, is_winter=True):
        """Processes one single hour of physics."""
        # 1. Update heating/cooling setpoints from profiles
        idx_168 = t % 168
        if self.h_setpoint_profile is not None:
            self.heating_setpoint = self.h_setpoint_profile[idx_168]
        if self.c_setpoint_profile is not None:
            self.cooling_setpoint = self.c_setpoint_profile[idx_168]

        # 2. Update ventilation trigger
        # Zone 2 keeps the 'vent_cooling_setpoint' from __init__
        # All other zones calculate it dynamically based on the current hour
        if self.zone_name == "Zone 2":
            # Ensure the Atrium always uses its fixed 27.0 value
            self.vent_cooling_setpoint = 27.0 
        else:
            # Everyone else uses the dynamic link
            self.vent_cooling_setpoint = max(self.vent_comfort_limit, self.cooling_setpoint - 0.1)

        # --- PRE-LOOP CALCULATIONS (now inside step) ---
        u_value_walls = self.calculate_nta8800_u_value(self.rc_facade)        
        u_value_roof  = self.calculate_nta8800_u_value(self.rc_roof) 
        u_value_ground = self.calculate_nta8800_u_value(self.rc_ground_floor)       

        trans_e_windows = self.area_windows * self.u_value_windows
        trans_e_walls = self.area_walls * u_value_walls
        trans_e_roof = self.area_roof * u_value_roof
        H_e_transmission = trans_e_walls + trans_e_roof + trans_e_windows

        # --- DATA FETCHING ---
        row = self.hourly_df.iloc[t]
        idx_168 = t % 168
        current_occ_val = self.occ_profile[idx_168]
        current_light_val = self.equip_profile[idx_168]
        current_vent_val = self.vent_profile[idx_168]
        current_sys = self.system_profile[idx_168]
        current_temp = self.current_temp

        # NEW: Logic for Cascaded Air Source
        t_ext = row['T'] 
        if t_source is None:
            t_source = t_ext  # Default to outdoor air if no buffer is specified. If specified, t_source is the temperature of the Atrium = pre-conditioned (buffer zone).

        # --- SOLAR GAINS ---
        glazing_gains_breakdown = {}
        opaque_gains_breakdown = {}

        for orient in self.solar_orientations:
            if current_temp >= self.vent_cooling_setpoint and row[orient] > 0:
                active_shading = self.solar_heat_coefficient_shading
            else:
                active_shading = 1.0

            glazing_val = (row[orient] * self.glass_areas[orient] * self.solar_heat_coefficient_glazing * active_shading)
            glazing_gains_breakdown[orient] = glazing_val

            opaque_val = (row[orient] * self.opaque_wall_areas[orient] * self.solar_absorption_coefficient * (u_value_walls / self.alfao))
            opaque_gains_breakdown[orient] = opaque_val

        sun_glazing = sum(glazing_gains_breakdown.values())
        sun_opaque_walls = sum(opaque_gains_breakdown.values())
        sun_opaque_roof = row['Horizontal'] * self.solar_absorption_coefficient * (u_value_roof / self.alfao) * self.area_roof

        # --- INTERNAL GAINS ---
        people_heat = self.max_people * current_occ_val * self.heat_per_person
        lighting_heat = self.lighting_w_m2 * self.floor_area * current_light_val
        equipment_heat = self.appliances_w_m2 * self.floor_area * current_light_val
        fans_heat = 0.5 * self.fans_power * current_vent_val                                        # Assuming 50% of fan power converts to heat in the space 
       
        total_internal_gains = people_heat + equipment_heat + lighting_heat + (fans_heat * 0.5)     # The excel adds another 0.5 factor (not sure why?)

        # --- Floor Heating Physics ---
        q_floor_gain = 0.0
        if self.floor_heating_activated:
            # Heating (warm air rises) = 11.0 | Cooling (cold air stays low) = 7.0
            u_eff = self.u_floor_heating if is_winter else 7.0
            t_water = self.t_floor_water_heating if is_winter else self.t_floor_water_cooling
    
            # --- Use input variables instead of hard-coded numbers ---
            # Heating mode logic: stop if room is above your heating limit
            if is_winter and self.current_temp > self.t_floor_limit_heating:
                q_floor_gain = 0.0
            # Cooling mode logic: stop if room is below your cooling limit
            elif not is_winter and self.current_temp < self.t_floor_limit_cooling:
                q_floor_gain = 0.0
            else:
                # Standard physics calculation
                q_floor_gain = u_eff * self.floor_heating_area * (t_water - self.current_temp)

        # --- the Cascaded Strategy (Atrium) ---
        q_atrium_effect = 0.0
        self.coil_UA = 0.5 
        if self.zone_name != "Zone 2" and t_source is not None:
            delta_t_atrium = t_source - 21.0 # this is the fixed setpoint
            q_atrium_effect = self.coil_UA * delta_t_atrium

        # --- TOTAL FREE GAINS (Including External Flow from Orchestrator) ---
        total_free_gains = sun_glazing + sun_opaque_walls + sun_opaque_roof + total_internal_gains + external_q_flow + q_floor_gain + q_atrium_effect

        # --- GROUND AND INFILTRATION ---
        flow_rate_inf = self.room_volume * self.infiltration_ach

        h_inf = (flow_rate_inf * self.air_density * self.air_heat_capacity) / 3600
        h_ground = self.area_ground * u_value_ground

        # --- VENTILATION ---
        current_flow_m3h = self.flow_rate_vent * current_occ_val * current_vent_val 

        h_vent_heat = (1 - self.eta) * (current_flow_m3h * self.air_density * self.air_heat_capacity) / 3600    # NEW LOGIC
        h_vent_cool = (current_flow_m3h + (self.natural_vent_rate * self.room_volume)) * self.air_density * self.air_heat_capacity / 3600 # NEW LOGIC

        # --- TOTAL HEAT TRANSFER ---
        # Facade and Infiltration always look at the outdoor air (t_ext)
        # Ventilation looks at the supply air (t_source)
        
        H_total_heat = H_e_transmission + h_vent_heat + h_inf        
        H_total_cool = H_e_transmission + h_vent_cool + h_inf  

        # Total exterior energy calculations split between facade (t_ext) and vent (t_source)
        Total_exterior_heat = ((H_e_transmission + h_inf) * t_ext) + (h_ground * self.t_ground) + (h_vent_heat * t_source)
        Total_exterior_cool = ((H_e_transmission + h_inf) * t_ext) + (h_ground * self.t_ground) + (h_vent_cool * t_source)

        H_total_with_ground_heat = H_total_heat + h_ground
        H_total_with_ground_cool = H_total_cool + h_ground
        
        # --- DYNAMIC VENTILATION LOGIC ---
        den_test = 1 + (self.dt / self.thermal_capacity) * H_total_with_ground_heat

        t_check = (current_temp + (self.dt / self.thermal_capacity) * (total_free_gains + Total_exterior_heat)) / den_test

        t_eq_heat = (total_free_gains + Total_exterior_heat) / H_total_with_ground_heat
        t_eq_cool = (total_free_gains + Total_exterior_cool) / H_total_with_ground_cool

        if t_check > self.vent_cooling_setpoint and t_ext < current_temp:
            H_active = H_total_with_ground_cool
            self.current_ach = (current_flow_m3h / self.room_volume) + self.natural_vent_rate 
        else:
            H_active = H_total_with_ground_heat
            self.current_ach = (current_flow_m3h / self.room_volume) 

        # --- EXPONENTIALS ---
        eps = 1e-9 # Safeguard: eps prevents division-by-zero glitches in high-insulation scenarios

        # Calculate the raw factors using the exponential decay formula
        exp_factor_heat_raw = 1 - np.exp(-(H_total_with_ground_heat) / self.thermal_capacity * self.dt)        # Or combine it into the following: exp_factor_active = 1 - np.exp(-(H_active) / self.thermal_capacity * self.dt) 
        exp_factor_cool_raw = 1 - np.exp(-(H_total_with_ground_cool) / self.thermal_capacity * self.dt)        # Or combine it into the following: exp_factor_active = 1 - np.exp(-(H_active) / self.thermal_capacity * self.dt)

        # Apply the floor: the factor can never be smaller than epsilon
        exp_factor_heat = max(exp_factor_heat_raw, eps)
        exp_factor_cool = max(exp_factor_cool_raw, eps)
        
        # --- NUMERICAL STABILITY CHECK & SCOUTING TEMPERATURES ---
        h_limit = 0.01  # Stability threshold [W/K]

        # Free Floating Heating scouting
        if H_total_with_ground_heat > h_limit:
            t_room_free_heating = current_temp + (t_eq_heat - current_temp) * exp_factor_heat
            t_room_max_heating = t_room_free_heating + (self.heating_power_max / H_total_with_ground_heat) * exp_factor_heat
        else:
            # Linear physics fallback: dT = (Power / Capacity) * time
            t_room_free_heating = current_temp + ((total_free_gains + Total_exterior_heat) / self.thermal_capacity) * self.dt
            t_room_max_heating = t_room_free_heating + (self.heating_power_max / self.thermal_capacity) * self.dt

        # Free Floating Cooling scouting
        if H_total_with_ground_cool > h_limit:
            t_room_free_cooling = current_temp + (t_eq_cool - current_temp) * exp_factor_cool
            t_room_max_cooling = t_room_free_cooling + (self.cooling_power_max / H_total_with_ground_cool) * exp_factor_cool
        else:
            t_room_free_cooling = current_temp + ((total_free_gains + Total_exterior_cool) / self.thermal_capacity) * self.dt
            t_room_max_cooling = t_room_free_cooling + (self.cooling_power_max / self.thermal_capacity) * self.dt

        # --- CONTROL LOGIC GATE ---
        q_heat_kwh = 0.0
        q_cool_kwh = 0.0
        p_needed = 0.0

        system_active = current_sys

        if system_active > 0:
            # STEP 1: Heating Capacity Limit (Too cold even with max heat)
            if t_room_max_heating < self.heating_setpoint:
                current_temp = t_room_max_heating
                q_heat_kwh = self.heating_power_max / 1000.0

            # STEP 2: Heating Setpoint Reached (Modulating power)
            elif t_room_max_heating >= self.heating_setpoint and t_room_free_heating < self.heating_setpoint:
                current_temp = self.heating_setpoint
                p_needed = H_total_with_ground_heat * (self.heating_setpoint - t_room_free_heating) / exp_factor_heat
                q_heat_kwh = p_needed / 1000.0

            # STEP 3: Free Floating - Standard Ventilation (HVAC is idle)
            elif t_room_free_heating >= self.heating_setpoint and t_room_free_heating <= self.vent_cooling_setpoint:
                current_temp = t_room_free_heating

            # STEP 4: Ventilation Setpoint Reached (Windows/Bypass modulating to hold 23.9)
            elif t_room_free_heating > self.vent_cooling_setpoint and t_room_free_cooling < self.vent_cooling_setpoint:
                current_temp = self.vent_cooling_setpoint

            # STEP 5: Free Floating - High Ventilation (Night cooling active but not enough)
            elif t_room_free_cooling >= self.vent_cooling_setpoint and t_room_free_cooling <= self.cooling_setpoint:
                current_temp = t_room_free_cooling

            # STEP 6: Cooling Setpoint Reached (Modulating chiller)
            elif t_room_max_cooling <= self.cooling_setpoint and t_room_free_cooling > self.cooling_setpoint:
                current_temp = self.cooling_setpoint
                
                if t_ext < self.cooling_setpoint:
                    # Natural vent helps; use the higher coupling
                    p_needed = H_total_with_ground_cool * (self.cooling_setpoint - t_room_free_cooling) / exp_factor_cool
                else:
                    # Natural vent hurts; Excel logic typically assumes windows closed during active chilling
                    p_needed = H_total_with_ground_heat * (self.cooling_setpoint - t_room_free_heating) / exp_factor_heat
                
                q_cool_kwh = p_needed / 1000.0

            # STEP 7: Cooling Capacity Limit (Too hot even with max cooling)
            elif t_room_max_cooling > self.cooling_setpoint:
                current_temp = t_room_max_cooling
                q_cool_kwh = self.cooling_power_max / 1000.0
        else:
            # System is OFF (e.g., Night/Weekend schedule)
            current_temp = t_room_free_heating

        # --- Adding UFH kWh usage to the timestep demands per zone ------------------------------------------------------------------------------------
        
        # --- Consolidated Billing & Storage ---
        # 1. Calculate floor energy (kWh)
        floor_heat_kwh = max(q_floor_gain, 0) / 1000.0
        floor_cool_kwh = min(q_floor_gain, 0) / 1000.0

        # 2. Sum Total Demand (Mechanical AHU + Floor)
        q_heat_total = q_heat_kwh + floor_heat_kwh
        q_cool_total = q_cool_kwh + floor_cool_kwh

        # 3. Update state and annual results
        self.current_temp = current_temp
        self.annual_heating_results.append(q_heat_total)
        self.annual_cooling_results.append(q_cool_total)
        self.annual_temp_results.append(current_temp)

        # 4. Fill the Detailed Dictionary (FOR EXCEL EXPORT)
        d = self.detailed_results
        
        # Identity columns
        d["Hour"].append(t + 1)
        d["T_ext"].append(t_ext)
        
        # Physics columns (KEEPING ALL INTERMEDIATE VALUES)
        d["Trans window"].append(trans_e_windows)
        d["Trans wall"].append(trans_e_walls)
        d["Trans roof"].append(trans_e_roof)
        d["Transmission tot (W/K)"].append(H_e_transmission)
        d["Infiltration (W/K)"].append(h_inf)
        d["Ventilation heat mode (W/K)"].append(h_vent_heat)
        d["Ventilation cool mode (W/K)"].append(h_vent_cool)
        d["Heating mode tot coupling (W/K)"].append(H_total_with_ground_heat)
        d["Cooling mode tot coupling (W/K)"].append(H_total_with_ground_cool)
        d["Ground coupling (W/K)"].append(h_ground)
        d["Solar glazing (Wh)"].append(sun_glazing)
        d["Solar opaque (Wh)"].append(sun_opaque_walls + sun_opaque_roof)
        d["People (Wh)"].append(people_heat)
        d["Lighting (Wh)"].append(lighting_heat)
        d["Appliances (Wh)"].append(equipment_heat)
        d["Fans (Wh)"].append(fans_heat)
        d["Internal heat gains (Wh)"].append(total_internal_gains)
        d["T_i Max heating"].append(t_room_max_heating)
        d["T_i free floating with H_heat"].append(t_room_free_heating)
        d["T_i Max cooling"].append(t_room_max_cooling)
        d["T_i free floating with H_cool"].append(t_room_free_cooling)
        
        # Final Zone Performance (Including UFH)
        d["T_i indoor (oC)"].append(current_temp)
        d["Q_heat (kWh)"].append(q_heat_total)
        d["Q_cool (kWh)"].append(q_cool_total)
        
        # Schedules
        d["Sch_Occupancy"].append(current_occ_val)
        d["Sch_Equipment"].append(current_light_val)
        d["Sch_SystemActive"].append(current_sys)
        d["Sch_Ventilation"].append(current_vent_val)
        
        # Solar orientation breakdown
        for orient in self.solar_orientations:
            d[f"G_{orient}_(Wh)"].append(glazing_gains_breakdown.get(orient, 0))
            d[f"O_{orient}_(Wh)"].append(opaque_gains_breakdown.get(orient, 0))

        return current_temp

    def run_simulation(self):
        """Can still be used for standalone tests of one zone."""
        # Reset results
        self.annual_heating_results = []
        self.annual_cooling_results = []
        self.annual_temp_results = []
        
        for t in range(len(self.hourly_df)):
            self.calculate_hour_step(t, external_q_flow=0)

        # --- OUTPUT SECTION --- 
        print(f"\n--- Results for {self.zone_name} ---")
        print(f"Simulation Complete.")
        print(f"Room Volume: {self.room_volume:.2f} m3")
        print(f"Peak Heating: {max(self.annual_heating_results):.2f} kW | Peak Cooling: {min(self.annual_cooling_results):.2f} kW")
        print(f"Total Annual Heat Demand: {sum(self.annual_heating_results):.2f} kWh")
        print(f"Total Annual Cool Demand: {sum(self.annual_cooling_results):.2f} kWh")

        return sum(self.annual_heating_results), sum(self.annual_cooling_results)
    
    def get_hourly_dataframe(self):
        """Returns the full thermal history as a DataFrame."""
        return pd.DataFrame(self.detailed_results)